# Zero-shot Bangla Medical VQA on BanglaMedVQA: MedGemma 4B-it
Follows the BanglaMedVQA paper (ACL 2026 Findings): open-ended answers, English zero-shot system prompt, Bangla question;
metrics Accuracy (exact match), BERTScore and LAVE (GPT-4.1-mini judge).
500 questions from the train splits: 5 categories x 2 sources (ChestX-ray8, MedICaT) x 50, seed 42.

**Secrets needed:** `HF_TOKEN` (accept the Gemma/MedGemma licences on Hugging Face) and `OPENAI_API_KEY`.
Restart the runtime between models.

## 1. Setup, Hugging Face login

In [1]:
!pip install -q -U transformers accelerate datasets qwen-vl-utils bert-score openai scikit-learn

def get_secret(name):
    try:                                    # Colab
        from google.colab import userdata
        return userdata.get(name)
    except ImportError:                     # Kaggle
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)

from huggingface_hub import login
login(get_secret("HF_TOKEN"))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 38.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


## 2. Config

In [2]:
import os, re, math, json, time, hashlib, unicodedata
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from datasets import load_dataset

MODEL_ID = "google/medgemma-4b-it"
TAG = "medgemma_4b"                              # used in output file names
SEED = 42
N_PER_CELL = 50                            # per category per source -> 5 x 2 x 50 = 500
MAX_NEW_TOKENS = 96
DATASET = "iiCEMAN/BanglaMedVQA"
SPLITS = {"chest_xray": "chest_xray_train", "medicat": "medicat_train"}
CATEGORIES = ["abnormality", "condition", "modality", "organ", "position"]

JUDGE_MODEL = "gpt-4.1-mini"               # same LAVE judge as the BanglaMedVQA paper

SAMPLE_CSV = "bnvqa_sample.csv"            # IDs only, same file for all four models
OUT_CSV    = f"bnvqa_results_{TAG}.csv"
SCORES_CSV = f"bnvqa_scores_{TAG}.csv"
torch.manual_seed(SEED)

## 3. Load data (Bangla columns are used for the prompt; English answers only to label Yes/No for sampling)

In [3]:
ds = {src: load_dataset(DATASET, split=split) for src, split in SPLITS.items()}

frames = []
for src, d in ds.items():
    m = d.remove_columns("image").to_pandas()      # metadata only; images are loaded per row later
    m["source"] = src
    m["src_idx"] = np.arange(len(m))
    frames.append(m)
meta = pd.concat(frames, ignore_index=True)
print(meta.groupby(["source", "category"]).size().unstack(0))

README.md:   0%|          | 0.00/6.04k [00:00<?, ?B/s]

data/chest_xray_train-00000-of-00003.par(…): reconstructing file:   0%|          |  0.00B /  473MB            

data/chest_xray_train-00000-of-00003.par(…): downloading bytes:           |  0.00B            

data/chest_xray_train-00001-of-00003.par(…): reconstructing file:   0%|          |  0.00B /  471MB            

data/chest_xray_train-00001-of-00003.par(…): downloading bytes:           |  0.00B            

...

Generating chest_xray_train split:   0%|          | 0/3500 [00:00<?, ? examples/s]

Generating chest_xray_test split:   0%|          | 0/200 [00:00<?, ? examples/s]

Generating medicat_train split:   0%|          | 0/3500 [00:00<?, ? examples/s]

Generating medicat_test split:   0%|          | 0/200 [00:00<?, ? examples/s]

source       chest_xray  medicat
category                        
abnormality         710      760
condition           707      443
modality            707      778
organ               695      757
position            681      762


## 4. Data check
Counts per source x category, abnormality Yes/No balance, and MedICaT answer formats.

In [4]:
def yes_no(ans_en):
    s = str(ans_en).strip().lower()
    if s.startswith("yes"): return "yes"
    if s.startswith("no") and not s.startswith(("none", "not ")): return "no"
    return None

meta["yn"] = meta["answer_en"].map(yes_no)
ab = meta[meta.category == "abnormality"]
print("Abnormality Yes/No per source:")
print(ab.groupby(["source", "yn"], dropna=False).size().unstack(0), "\n")

for cat in CATEGORIES:
    print(f"--- medicat / {cat}: top answers ---")
    print(meta[(meta.source == "medicat") & (meta.category == cat)].answer_en.value_counts().head(8), "\n")

Abnormality Yes/No per source:
source  chest_xray  medicat
yn                         
no           301.0    103.0
yes          409.0    642.0
NaN            NaN     15.0 

--- medicat / abnormality: top answers ---
answer_en
Yes    496
yes    146
no     102
1       15
No       1
Name: count, dtype: int64 

--- medicat / condition: top answers ---
answer_en
no finding                   174
tumor                          5
mass                           5
lesion                         3
signal hyperintensities        2
hemorrhage                     2
anterior mediastinal mass      2
large mass                     2
Name: count, dtype: int64 

--- medicat / modality: top answers ---
answer_en
CT            386
MRI           250
X-ray         119
Ultrasound      9
Other           8
US, CT          1
MR, CT          1
SPECT, CT       1
Name: count, dtype: int64 

--- medicat / organ: top answers ---
answer_en
brain                      300
chest                      268
abdomen          

## 5. Sampling (identical in all four notebooks)
50 per category per source; abnormality balanced 25 Yes / 25 No per source; one image per category. The sample hash must match across notebooks.

In [5]:
picked = []
for src in SPLITS:
    for cat in CATEGORIES:
        pool = meta[(meta.source == src) & (meta.category == cat)]
        pool = pool.sample(frac=1, random_state=SEED).drop_duplicates("image_id")   # one image per category
        if cat == "abnormality":
            pool = pool[pool.yn.notna()]
            yes, no = pool[pool.yn == "yes"], pool[pool.yn == "no"]
            n_no = min(N_PER_CELL // 2, len(no))
            n_yes = N_PER_CELL - n_no
            if n_no < N_PER_CELL // 2:
                print(f"WARNING {src}/abnormality: only {len(no)} 'No' rows, using {n_no} No + {n_yes} Yes")
            cell = pd.concat([yes.head(n_yes), no.head(n_no)])
        else:
            cell = pool.head(N_PER_CELL)
        assert len(cell) == N_PER_CELL, f"{src}/{cat}: only {len(cell)} rows available"
        picked.append(cell)

sample = (pd.concat(picked).sort_values(["source", "category", "src_idx"])
          .reset_index(drop=True))
sample.insert(0, "idx", np.arange(len(sample)))

ids = "|".join(f"{s}:{i}" for s, i in zip(sample.source, sample.src_idx))
SAMPLE_HASH = hashlib.md5(ids.encode()).hexdigest()[:12]
print("Sample size:", len(sample), "| sample hash:", SAMPLE_HASH, "(must be identical in all 4 notebooks)")

id_cols = ["idx", "source", "src_idx", "image_id", "category"]
if os.path.exists(SAMPLE_CSV):
    old = pd.read_csv(SAMPLE_CSV)
    assert old[id_cols].astype(str).equals(sample[id_cols].astype(str)), "Sample differs from bnvqa_sample.csv!"
    print("Matches existing", SAMPLE_CSV)
else:
    sample[id_cols].to_csv(SAMPLE_CSV, index=False)
    print("Saved", SAMPLE_CSV)
print(sample.groupby(["source", "category"]).size().unstack(0))

Sample size: 500 | sample hash: caed7502c8c2 (must be identical in all 4 notebooks)
Saved bnvqa_sample.csv
source       chest_xray  medicat
category                        
abnormality          50       50
condition            50       50
modality             50       50
organ                50       50
position             50       50


## 6. Load model: MedGemma 4B-it

In [6]:
from transformers import AutoProcessor, AutoModelForImageTextToText

model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")
processor = AutoProcessor.from_pretrained(MODEL_ID)
model.eval()
print(model.device)

config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

cuda:0


## 7. Prompt (paper's zero-shot prompt) and generation
Greedy decoding.

In [7]:
# Zero-shot system prompt from the BanglaMedVQA paper (Appendix C), verbatim.
SYSTEM_PROMPT = """You are a careful, clinically grounded medical vision-language assistant.

Task: You will be given a medical image and a single question about that image (e.g., modality, organ, abnormality, condition, or the position of the condition). Your job is to look at the image and provide the exact, minimal answer to the question.

CRITICAL: You must respond with ONLY the exact answer. No explanations, no sentences, no extra words.

STRICT OUTPUT RULES:
- ONE word or short phrase only
- NO sentences or explanations
- NO "This is..." or "The image shows..."
- NO punctuation unless part of the answer"""

def get_image(r):
    return ds[r["source"]][int(r["src_idx"])]["image"].convert("RGB")

def build_messages(image, question_bn):
    return [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [{"type": "image", "image": image},
                                     {"type": "text", "text": question_bn}]},
    ]

In [8]:
@torch.no_grad()
def generate(image, question_bn):
    messages = build_messages(image, question_bn)
    inputs = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                                           return_dict=True, return_tensors="pt").to(model.device, dtype=torch.bfloat16)
    n = inputs["input_ids"].shape[-1]
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    return processor.decode(out[0][n:], skip_special_tokens=True).strip()

In [9]:
def clean(raw):
    """Minimal cleaning, same for every model: <answer> tag, whitespace, wrapping quotes, trailing ।/."""
    s = str(raw) if raw is not None and not (isinstance(raw, float) and math.isnan(raw)) else ""
    m = re.search(r"<answer>(.*?)(</answer>|$)", s, re.S)
    if m:
        s = m.group(1)
    s = s.strip().strip('"\'“”‘’`').strip()
    s = re.sub(r"[।.\s]+$", "", s)
    return s

def run_one(r):
    raw = generate(get_image(r), r["question_bn"])
    return raw, clean(raw)

## 8. Smoke test (5 questions, one per category; not saved)

In [10]:
smoke = sample.groupby("category").head(1).head(5)
for _, r in smoke.iterrows():
    t = time.time()
    raw, pred = run_one(r)
    print(f"[{r.source}/{r.category}] true={r.answer_bn!r} pred={pred!r} raw={raw!r} ({time.time()-t:.1f}s)")

[chest_xray/abnormality] true='হ্যাঁ' pred='হ্যাঁ' raw='হ্যাঁ' (5.9s)
[chest_xray/condition] true='প্রলেপ, অনুপ্রবেশ, গন্ড এবং অনুপ্রবেশ (মাঝখানে ডান দিকে)।' pred='Chest' raw='Chest' (4.0s)
[chest_xray/modality] true='এক্স-রে' pred='X-ray' raw='X-ray' (4.2s)
[chest_xray/organ] true='বুকের' pred='Chest' raw='Chest' (4.0s)
[chest_xray/position] true='ডানদিকের মাঝখানে' pred='Lung' raw='Lung' (4.1s)


## 9. Run the 500 questions (resumes if the session restarts)

In [11]:
done = pd.read_csv(OUT_CSV) if os.path.exists(OUT_CSV) else pd.DataFrame()
rows = done.to_dict("records")
seen = set(done["idx"]) if len(done) else set()

for _, r in tqdm(sample.iterrows(), total=len(sample)):
    if r["idx"] in seen:
        continue
    raw, pred = run_one(r)
    rows.append({"idx": r["idx"], "image_id": r["image_id"], "source": r["source"],
                 "category": r["category"], "question": r["question_bn"],
                 "true_answer": r["answer_bn"], "predicted_answer": pred, "raw_output": raw})
    if len(rows) % 25 == 0:
        pd.DataFrame(rows).to_csv(OUT_CSV, index=False)

results = pd.DataFrame(rows).sort_values("idx").reset_index(drop=True)
results.to_csv(OUT_CSV, index=False)
print(len(results), "rows saved to", OUT_CSV)

  0%|          | 0/500 [00:00<?, ?it/s]

500 rows saved to bnvqa_results_medgemma_4b.csv


## 10. Accuracy, normalized accuracy, BERTScore, output-failure flags

In [12]:
BN_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
YES = {"হ্যাঁ", "হ্যা", "হাঁ", "yes"}
NO = {"না", "no"}

def norm(s):
    """Supplementary normalization (ours, not the paper's)."""
    s = unicodedata.normalize("NFC", str(s)).lower().translate(BN_DIGITS)
    s = "".join(ch for ch in s if not unicodedata.category(ch).startswith("P"))
    s = re.sub(r"\s+", " ", s).strip()
    if s in YES: return "yes"
    if s in NO: return "no"
    return s

results = pd.read_csv(OUT_CSV).sort_values("idx").reset_index(drop=True)
results["predicted_answer"] = results["predicted_answer"].fillna("").astype(str)
results["true_answer"] = results["true_answer"].astype(str)

# Accuracy (paper): exact match after the same minimal cleaning on both sides
results["correct"] = results.true_answer.map(clean) == results.predicted_answer
# Normalized accuracy (supplementary)
results["correct_norm"] = results.true_answer.map(norm) == results.predicted_answer.map(norm)

# BERTScore F1 (paper), lang="bn" -> multilingual BERT (mBERT, layer 9, no idf, no rescale).
# Own implementation of the bert-score defaults: the bert-score package crashes on transformers v5.
import torch
from transformers import AutoTokenizer, AutoModel

@torch.no_grad()
def bertscore_f1(cands, refs, model_name="bert-base-multilingual-cased", layer=9, bs=32,
                 tok=None, mdl=None):
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    tok = tok or AutoTokenizer.from_pretrained(model_name)
    if mdl is None:
        mdl = AutoModel.from_pretrained(model_name)
        mdl.encoder.layer = mdl.encoder.layer[:layer]
    mdl = mdl.to(dev).eval()

    def embed(texts):
        out = []
        for i in range(0, len(texts), bs):
            enc = tok(texts[i:i + bs], return_tensors="pt", padding=True,
                      truncation=True, max_length=512).to(dev)
            h = torch.nn.functional.normalize(mdl(**enc).last_hidden_state.float(), dim=-1)
            for j in range(h.size(0)):
                n = int(enc["attention_mask"][j].sum())
                out.append(h[j, :n].cpu())               # keep [CLS]/[SEP] as match targets
        return out

    C, R = embed(list(cands)), embed(list(refs))
    f1 = []
    for c, r in zip(C, R):
        if len(c) <= 2 or len(r) <= 2:                     # empty text (only [CLS][SEP])
            f1.append(0.0); continue
        sim = r @ c.T                                      # ref tokens x cand tokens
        # as in bert_score (idf=False): [CLS]/[SEP] get zero weight in the averages
        # but still count as possible match targets
        rec = sim.max(1).values[1:-1].mean()
        prec = sim.max(0).values[1:-1].mean()
        f1.append(float(2 * prec * rec / (prec + rec)))
    return torch.tensor(f1)

preds = [p if p.strip() else " " for p in results.predicted_answer]
F1 = bertscore_f1(preds, results.true_answer.tolist())
results["bertscore_f1"] = F1.numpy()
results.loc[results.predicted_answer.str.strip() == "", "bertscore_f1"] = 0.0

# Output failure flags
has_bn = results.predicted_answer.str.contains(r"[ঀ-৿]")
has_latin = results.predicted_answer.str.contains(r"[A-Za-z]")
results["is_empty"] = results.predicted_answer.str.strip() == ""
results["is_english"] = has_latin & ~has_bn
results["is_long"] = results.predicted_answer.str.split().str.len() > 6

results.to_csv(OUT_CSV, index=False)
print(f"Accuracy:            {results.correct.mean():.3f}")
print(f"Normalized accuracy: {results.correct_norm.mean():.3f}")
print(f"BERTScore F1:        {results.bertscore_f1.mean():.3f}")
print(results[["is_empty", "is_english", "is_long"]].mean().round(3))

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Accuracy:            0.016
Normalized accuracy: 0.122
BERTScore F1:        0.714
is_empty      0.000
is_english    0.982
is_long       0.000
dtype: float64


## 11. LAVE (GPT-4.1-mini judge, paper's prompt, one pair per call, temperature 0; resumes)

In [13]:
from openai import OpenAI
client = OpenAI(api_key=get_secret("OPENAI_API_KEY"))

# LAVE judge prompt from the BanglaMedVQA paper (Appendix C)
LAVE_SYSTEM = ("You are a strict evaluator. "
               "Compare each prediction with its reference. "
               "Return ONLY a valid JSON array of floats between 0 and 1. "
               "Each float represents similarity between reference and prediction. "
               "If a reference answer is missing, treat it as 'No'. "
               "Do NOT return any text outside the JSON.")

def lave(ref, pred, retries=3):
    msg = f"Reference: {ref}\nPrediction: {pred}"
    for _ in range(retries):
        try:
            r = client.chat.completions.create(
                model=JUDGE_MODEL, temperature=0,
                messages=[{"role": "system", "content": LAVE_SYSTEM},
                          {"role": "user", "content": msg}])
            out = r.choices[0].message.content
            val = json.loads(re.search(r"\[.*?\]", out, re.S).group())[0]
            return min(max(float(val), 0.0), 1.0)
        except Exception as e:
            last = e
            time.sleep(2)
    print("LAVE failed:", type(last).__name__)
    return None            # failed calls stay empty, not counted as 0

results = pd.read_csv(OUT_CSV)
if "lave" not in results:
    results["lave"] = np.nan
todo = results.index[results.lave.isna()]
for n, i in enumerate(tqdm(todo)):     # resumes: only rows without a score are sent
    results.at[i, "lave"] = lave(results.at[i, "true_answer"], results.at[i, "predicted_answer"])
    if (n + 1) % 25 == 0:
        results.to_csv(OUT_CSV, index=False)
results.to_csv(OUT_CSV, index=False)

print("LAVE failures:", int(results.lave.isna().sum()))
print(f"LAVE: {results.lave.mean() * 100:.2f}%")

  0%|          | 0/500 [00:00<?, ?it/s]

LAVE failures: 0
LAVE: 31.48%


### LAVE manual check (20 position/condition rows)
If left vs right or different conditions get high scores, flag it in the report.

In [14]:
# Manual check: does the judge separate left/right and different conditions?
chk = results[results.category.isin(["position", "condition"])].sample(20, random_state=SEED)
pd.set_option("display.max_colwidth", 60)
chk[["category", "true_answer", "predicted_answer", "lave"]]

,category,true_answer,predicted_answer,lave
245,position,কোন অবস্থা চিহ্নিত হয়নি।,Chest,0.0
65,condition,"কনসোলিডেশন (N/A), মধ্য ডান দিকে নিউমোনিয়া",Chest,0.0
80,condition,কোনও অবস্থা চিহ্নিত হয়নি।,Chest,0.0
458,position,পিত্তথলি,Kidney,0.0
328,condition,কিছু পাওয়া যায়নি,Liver,0.0
315,condition,খালি সেলা,Sinus,0.0
219,position,কোন নির্দিষ্ট অবস্থা চিহ্নিত হয়নি।,Chest,0.0
470,position,"মস্তিষ্ক, বাম প্যারাইটাল লোব",মাথা,0.2
474,position,মেরুদণ্ড।,Brain,0.0
95,condition,"কারডিওমেগালি (নীচের বাম দিকে), ইনফিলট্রেশন (প্রযোজ্য নয়)",Heart,0.0


## 12. Scores and tables
Saved to `bnvqa_scores_<TAG>.csv`. `bnvqa_results_<TAG>.csv` holds every row with all metrics.

In [15]:
from sklearn.metrics import classification_report, confusion_matrix

def wilson(k, n, z=1.96):
    if n == 0: return (0, 0)
    p = k / n; d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    h = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return c - h, c + h

def summarize(g):
    n, k = len(g), int(g.correct.sum())
    lo, hi = wilson(k, n)
    return pd.Series({"n": n, "accuracy": k / n, "acc_ci_low": lo, "acc_ci_high": hi,
                      "accuracy_norm": g.correct_norm.mean(), "bertscore": g.bertscore_f1.mean(),
                      "lave": g.lave.mean(), "lave_failures": int(g.lave.isna().sum())})

results = pd.read_csv(OUT_CSV)
by_cell = results.groupby(["source", "category"]).apply(summarize).reset_index()
by_src = results.groupby("source").apply(summarize).reset_index().assign(category="all")
overall = summarize(results).to_frame().T.assign(source="all", category="all")
# Paper's "Overall (Avg.)" = mean of the two sources
avg = by_src[["accuracy", "accuracy_norm", "bertscore", "lave"]].mean().to_frame().T.assign(
    source="avg_of_sources", category="all", n=len(results))
scores = pd.concat([by_cell, by_src, overall, avg], ignore_index=True)
scores.insert(0, "model", TAG)
scores.to_csv(SCORES_CSV, index=False)

pct = ["accuracy", "accuracy_norm", "bertscore", "lave"]
show = scores.copy(); show[pct] = (show[pct] * 100).round(2)
print("Table 1 (paper layout): Acc / BScore / LAVE per source")
print(show[show.category == "all"][["source", "n"] + pct].to_string(index=False), "\n")
print("Table 2 (paper layout): per-category LAVE")
print(show[show.category != "all"].pivot(index="category", columns="source", values="lave"), "\n")

# Abnormality: Yes/No answer distribution and per-class metrics
ab = results[results.category == "abnormality"].copy()
ab["true_yn"] = ab.true_answer.map(norm)
ab["pred_yn"] = ab.predicted_answer.map(norm).where(lambda s: s.isin(["yes", "no"]), "other")
print("Abnormality predicted answer distribution:")
print(ab.groupby("source").pred_yn.value_counts().unstack(fill_value=0), "\n")
print(classification_report(ab.true_yn, ab.pred_yn, labels=["yes", "no"], zero_division=0))
print(pd.DataFrame(confusion_matrix(ab.true_yn, ab.pred_yn, labels=["yes", "no", "other"]),
                   index=["true yes", "true no", "true other"], columns=["pred yes", "pred no", "pred other"]))
print("\nOutput failure rates per source:")
print(results.groupby("source")[["is_empty", "is_english", "is_long"]].mean().round(3))
print("\nSaved", SCORES_CSV, "and", OUT_CSV)

Table 1 (paper layout): Acc / BScore / LAVE per source
        source     n  accuracy  accuracy_norm  bertscore  lave
    chest_xray 250.0       1.2           12.0      71.46 36.44
       medicat 250.0       2.0           12.4      71.29 26.52
           all 500.0       1.6           12.2      71.38 31.48
avg_of_sources 500.0       1.6           12.2      71.38 31.48 

Table 2 (paper layout): per-category LAVE
source       chest_xray  medicat
category                        
abnormality         6.0      4.0
condition           9.8     12.4
modality           84.8     61.8
organ              78.4     47.8
position            3.2      6.6 

Abnormality predicted answer distribution:
pred_yn     no  other  yes
source                    
chest_xray  12      9   29
medicat      7      7   36 

              precision    recall  f1-score   support

         yes       0.60      0.78      0.68        50
          no       1.00      0.38      0.55        50

   micro avg       0.69      0.58   

/tmp/ipykernel_1043/937287166.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  by_cell = results.groupby(["source", "category"]).apply(summarize).reset_index()
/tmp/ipykernel_1043/937287166.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  by_src = results.groupby("source").apply(summarize).reset_index().assign(category="all")
